# 06 – Lấy ngẫu nhiên 150 ảnh để đo độ đồng thuận gán nhãn giữa các thành viên

**Mục đích:** tạo bộ ảnh để **mỗi thành viên gán nhãn lại độc lập, từ đầu** (không xem nhãn cũ, không trao đổi với
nhau). Notebook 07 sẽ so các bộ nhãn này với nhau và với nhãn hiện có của bộ dữ liệu để đo:
- độ đồng thuận phát hiện (ghép hộp theo IoU, F1 giữa hai người),
- độ đồng thuận vị trí (IoU trung bình của các hộp ghép được),
- độ đồng thuận lớp (Cohen's kappa từng cặp người, Fleiss' kappa cho cả ba người).

**Cách lấy mẫu:** lấy mẫu ngẫu nhiên đơn giản, không hoàn lại, `N_SAMPLE = 150` ảnh trong toàn bộ ảnh của
`cleaned_data_chuanhoa.zip` (dữ liệu sau notebook 01b), hạt giống cố định `SEED` để lấy lại được đúng bộ mẫu.
Mẫu ngẫu nhiên đơn giản cho ước lượng không thiên lệch về độ đồng thuận trên toàn bộ dữ liệu; notebook này in ra số
hộp, số lớp, số ảnh không có hộp trong mẫu để báo cáo trong bài. Không chọn ảnh theo tên file.

**Chống thiên lệch khi gán nhãn (gán nhãn mù):**
- Ảnh được đổi tên thành mã ngẫu nhiên `DT001.jpg … DT150.jpg` (thứ tự đã xáo trộn), không kèm nhãn cũ.
- Bảng mã → ảnh gốc (`KHONG_GUI/bang_ma.csv`) và nhãn cũ (`KHONG_GUI/nhan_goc/`) để ở thư mục riêng,
  **không gửi cho người gán nhãn**; chỉ notebook 07 dùng.
- Ảnh chép nguyên byte (không nén lại, không đổi kích thước).

**Đầu vào:** `MyDrive/VNTrafficSign/output_chuanhoa/cleaned_data_chuanhoa.zip`

**Đầu ra:** `MyDrive/VNTrafficSign/output_dongthuan/`
- `goi_gan_nhan.zip` – GỬI cho từng thành viên: `images/DT001.jpg …`, `classes.txt` (đúng tên 115 lớp, đúng thứ tự),
  `HUONG_DAN_GAN_NHAN.md`
- `KHONG_GUI/bang_ma.csv`, `KHONG_GUI/nhan_goc/DTxxx.txt`, `KHONG_GUI/data_goc.yaml` – chỉ dùng cho notebook 07
- `tom_tat_mau.json`, `phan_bo_lop_trong_mau.csv` – mô tả mẫu (đưa vào bài)

In [1]:
import hashlib
import json
import os
import random
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import yaml

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/VNTrafficSign')
    LOCAL_ROOT = Path('/content')
except ImportError:  # chạy ngoài Colab (kiểm thử)
    DRIVE_ROOT = Path(os.environ.get('VNTS_DRIVE_ROOT', './VNTrafficSign'))
    LOCAL_ROOT = Path(os.environ.get('VNTS_LOCAL_ROOT', './local'))

CLEAN_ZIP = DRIVE_ROOT / 'output_chuanhoa' / 'cleaned_data_chuanhoa.zip'
OUT = DRIVE_ROOT / 'output_dongthuan'
EXTRACT_DIR = LOCAL_ROOT / 'cleaned_data_dongthuan'

# ---- Tham số ----
N_SAMPLE = 150
SEED = 20261029            # cố định để lấy lại đúng bộ mẫu; ghi vào bài
PREFIX = 'DT'
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
try:
    display
except NameError:
    display = print
assert CLEAN_ZIP.exists(), f'Không tìm thấy {CLEAN_ZIP}'
if (OUT / 'KHONG_GUI' / 'bang_ma.csv').exists():
    raise SystemExit(f'⛔ Đã có bộ mẫu ở {OUT}. Không lấy lại để tránh đổi mẫu giữa chừng. '
                     'Muốn lấy lại thì tự xóa thư mục đó trước.')

Mounted at /content/drive


## 1. Giải nén, liệt kê ảnh và nhãn

In [2]:
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
with zipfile.ZipFile(CLEAN_ZIP) as zf:
    zf.extractall(EXTRACT_DIR)
yaml_files = list(EXTRACT_DIR.rglob('data.yaml'))
assert len(yaml_files) == 1, f'Cần đúng 1 data.yaml, thấy {len(yaml_files)}'
ROOT = yaml_files[0].parent
cfg = yaml.safe_load(open(yaml_files[0], encoding='utf-8'))
NAMES = list(cfg['names'].values()) if isinstance(cfg['names'], dict) else list(cfg['names'])
assert len(set(NAMES)) == len(NAMES), 'Tên lớp bị trùng → không ánh xạ theo tên được'

rows = []
for img_dir in sorted(ROOT.glob('*/images')):
    for p in sorted(img_dir.iterdir()):
        if p.suffix.lower() in IMG_EXTS:
            rows.append({'path': str(p), 'label': str(img_dir.parent / 'labels' / f'{p.stem}.txt')})
images = pd.DataFrame(rows).sort_values('path').reset_index(drop=True)   # thứ tự cố định trước khi lấy mẫu
print(f'{len(images)} ảnh trong bộ dữ liệu, {len(NAMES)} lớp')


def read_classes(label_path):
    if not Path(label_path).exists():
        return []
    return [int(float(l.split()[0])) for l in open(label_path, encoding='utf-8') if len(l.split()) >= 5]

14509 ảnh trong bộ dữ liệu, 115 lớp


## 2. Lấy mẫu ngẫu nhiên đơn giản và tạo gói gán nhãn mù

In [3]:
rng = random.Random(SEED)
idx = rng.sample(range(len(images)), N_SAMPLE)          # không hoàn lại; thứ tự lấy ra đã ngẫu nhiên → dùng làm mã
sample = images.iloc[idx].reset_index(drop=True)
sample['code'] = [f'{PREFIX}{i + 1:03d}' for i in range(N_SAMPLE)]

PACK = LOCAL_ROOT / 'goi_gan_nhan'
SECRET = OUT / 'KHONG_GUI'
for d in (PACK, ):
    shutil.rmtree(d, ignore_errors=True)
(PACK / 'images').mkdir(parents=True)
(SECRET / 'nhan_goc').mkdir(parents=True, exist_ok=True)

recs = []
for r in sample.itertuples():
    src = Path(r.path)
    dst = PACK / 'images' / f'{r.code}{src.suffix.lower()}'
    shutil.copyfile(src, dst)                                # nguyên byte
    lab = Path(r.label)
    (SECRET / 'nhan_goc' / f'{r.code}.txt').write_text(lab.read_text(encoding='utf-8') if lab.exists() else '',
                                                       encoding='utf-8')
    cls = read_classes(lab)
    recs.append({'code': r.code, 'anh_goc': str(src.relative_to(ROOT)),
                 'sha256': hashlib.sha256(src.read_bytes()).hexdigest(), 'so_hop_goc': len(cls)})
pd.DataFrame(recs).to_csv(SECRET / 'bang_ma.csv', index=False, encoding='utf-8-sig')
shutil.copyfile(yaml_files[0], SECRET / 'data_goc.yaml')

(PACK / 'classes.txt').write_text('\n'.join(NAMES) + '\n', encoding='utf-8')
(PACK / 'HUONG_DAN_GAN_NHAN.md').write_text(f"""# Hướng dẫn gán nhãn lại {N_SAMPLE} ảnh (đo độ đồng thuận)

1. **Làm một mình.** Không xem nhãn cũ của bộ dữ liệu, không hỏi/trao đổi với người khác cho tới khi cả ba nộp xong.
2. Gán nhãn **mọi biển báo** trong ảnh theo đúng quy tắc gán nhãn đã dùng cho bộ dữ liệu (hộp chữ nhật sát mép biển,
   tên lớp theo QCVN 41:2024). Ảnh không có biển báo nào thì để trống.
3. Tên lớp phải **chép đúng** từ `classes.txt` (đúng chữ, đúng dấu, đúng hoa/thường).
4. Gán nhãn bằng Roboflow (hoặc công cụ khác) như khi làm bộ dữ liệu:
   - Roboflow: tạo **project riêng của mình** (Object Detection), tải lên thư mục `images/`, gán nhãn.
   - Khi tạo Version: **không** chọn Preprocessing (bỏ cả Auto-Orient, Resize) và **không** chọn Augmentation;
     đưa toàn bộ ảnh vào train (không cần chia valid/test).
   - Export định dạng **YOLOv8**, tải file zip về.
5. Đổi tên file zip thành `nhan_<Tên>.zip` (ví dụ `nhan_Tram.zip`) và nộp cho người phụ trách. Ghi lại tổng thời gian làm.
""", encoding='utf-8')
OUT.mkdir(parents=True, exist_ok=True)
pack_zip = OUT / 'goi_gan_nhan.zip'
with zipfile.ZipFile(pack_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(PACK.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(PACK))
print('✅ Gói gửi người gán nhãn:', pack_zip)
print('⛔ KHÔNG gửi thư mục:', SECRET)

✅ Gói gửi người gán nhãn: /content/drive/MyDrive/VNTrafficSign/output_dongthuan/goi_gan_nhan.zip
⛔ KHÔNG gửi thư mục: /content/drive/MyDrive/VNTrafficSign/output_dongthuan/KHONG_GUI


## 3. Mô tả mẫu (đưa vào bài)

In [4]:
all_cls = [c for lab in images['label'] for c in read_classes(lab)]
smp_cls = [c for lab in sample['label'] for c in read_classes(lab)]
cnt = pd.Series(smp_cls, dtype=int).value_counts()
dist = pd.DataFrame({'class_id': range(len(NAMES)), 'class_name': NAMES,
                     'so_hop_trong_mau': [int(cnt.get(i, 0)) for i in range(len(NAMES))]})
dist.to_csv(OUT / 'phan_bo_lop_trong_mau.csv', index=False, encoding='utf-8-sig')
n_box_img = sample['label'].map(lambda l: len(read_classes(l)))
summary = {
    'n_anh_bo_du_lieu': int(len(images)), 'n_anh_mau': N_SAMPLE, 'seed': SEED,
    'phuong_phap': 'lấy mẫu ngẫu nhiên đơn giản không hoàn lại (random.Random(seed).sample) trên danh sách ảnh sắp theo đường dẫn',
    'n_hop_trong_mau': int(len(smp_cls)), 'n_hop_bo_du_lieu': int(len(all_cls)),
    'hop_moi_anh_mau_tb': round(float(n_box_img.mean()), 3), 'hop_moi_anh_bo_du_lieu_tb': round(len(all_cls) / len(images), 3),
    'n_anh_mau_khong_co_hop': int((n_box_img == 0).sum()),
    'n_lop_co_mat_trong_mau': int((dist['so_hop_trong_mau'] > 0).sum()), 'n_lop': len(NAMES),
}
json.dump(summary, open(OUT / 'tom_tat_mau.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
display(pd.Series(summary))

,0
n_anh_bo_du_lieu,14509
n_anh_mau,150
seed,20261029
phuong_phap,lấy mẫu ngẫu nhiên đơn giản không hoàn lại (ra...
n_hop_trong_mau,260
n_hop_bo_du_lieu,27150
hop_moi_anh_mau_tb,1.733
hop_moi_anh_bo_du_lieu_tb,1.871
n_anh_mau_khong_co_hop,0
n_lop_co_mat_trong_mau,68
